## **Tasks**
* In this coursework, you will implement Value Iteration, Policy Iteration and Q-Learning Iteration that plan/learn to play 3x3 Tic-Tac-Toe game. You will test your agents against other rule-based agents that are provided. You can also play against all the agents including your own agents to test them.
* A general framework for the game and agents is provided. Run each code cell below in order, when you see a <>

### Part 0: Environment: packages, constants, basic code

In [ ]:
import os

import numpy as np
import pickle
from abc import ABC, abstractmethod

# Constants for the game
EMPTY = 0
PLAYER_X = 1
PLAYER_O = -1
GAME_ROW, GAME_COL = 3, 3


#### 0.1: The ***Game*** class:

play(): simulate one game

*show_board* indicate whether the states are printed during play

In [ ]:
class Game:
    """
    Define the tictactoe game. The function and variable names should be self explained.

    @author: chenxy
    """
    def __init__(self, player_x, player_o, show_board=False):
        self.board = np.zeros((GAME_ROW, GAME_COL), dtype=int)
        self.player_x = player_x
        self.player_o = player_o
        self.current_player = self.player_x
        self.winner = None
        self.show_board = show_board
        self.turn = 0

    def get_empty_positions(self):
        return [(i, j) for i in range(GAME_ROW) for j in range(GAME_COL) if self.board[i, j] == EMPTY]

    def is_winner(self, player):
        symbol = player.symbol
        for i in range(GAME_ROW):
            if np.all(self.board[i, :] == symbol) or np.all(self.board[:, i] == symbol):
                return True
        if np.all(np.diag(self.board) == symbol) or np.all(np.diag(np.fliplr(self.board)) == symbol):
            return True
        return False

    def is_draw(self):
        return np.all(self.board != EMPTY)

    def make_move(self, position):
        if self.board[position] != EMPTY:
            # Don't raise an exception, just return indicating an invalid move
            return False
        self.board[position] = self.current_player.symbol
        return True

    def switch_player(self):
        self.current_player = self.player_x if self.current_player == self.player_o else self.player_o

    def get_hash(self, board=None):
        if board is None:
            board = self.board
        return ','.join(str(int(elem)) for elem in board.flatten())

    def reset(self):
        self.__init__(self.player_x, self.player_o, self.show_board)

    def is_terminal(self):
        # Check for a win in rows, columns, and diagonals
        for i in range(GAME_ROW):
            if np.all(self.board[i] == self.current_player.symbol) or \
               np.all(self.board[:, i] == self.current_player.symbol):
                return True
        if np.all(np.diag(self.board) == self.current_player.symbol) or \
           np.all(np.diag(np.fliplr(self.board)) == self.current_player.symbol):
            return True
        # Check for a draw (no empty positions left)
        if not np.any(self.board == EMPTY):
            return True
        return False

    def play(self):
        self.reset()
        while True:
          position = self.current_player.move(self)
          if not self.make_move(position):  # If move is invalid, skip to next turn
            raise Exception("Something is wrong! No empty positions now!")
          self.turn+=1

          if self.is_terminal():
              if self.is_winner(self.current_player):
                  self.winner = self.current_player.symbol
                  print(f"Player {self.current_player.symbol} wins!")
                  break  # Exit the loop immediately after a win

              if self.is_draw():
                  print("It's a draw!")
                  break  # Exit the loop immediately after a draw

          if self.show_board:
              print(f"Turn {self.turn}: Player {self.current_player.symbol}")
              self.print_board()

          self.switch_player()

        if self.show_board:
            self.print_board()  # Show the final board state

    def print_board(self):
        symbols = {EMPTY: ' ', PLAYER_X: 'X', PLAYER_O: 'O'}
        for i in range(GAME_ROW):
            print('|' + '|'.join(symbols[s] for s in self.board[i]) + '|')
        print()

#### 0.2 Agent abstract class, all *agents* class inherit this one.
* *RandomAgent*: perform random action
* *AggressiveAgent*: choose the winning action
* *DefensiveAgent*: stop opponent's winning action

In [ ]:
class Agent(ABC):
    def __init__(self, symbol):
        self.symbol = symbol
        self.states_value = {}  # State values used by ValueIterationAgent

    @abstractmethod
    def move(self, game):
        pass

    def save_policy(self, file_name):
        with open(file_name, 'wb') as f:
            pickle.dump(self.states_value, f)

    def load_policy(self, file_name):
        with open(file_name, 'rb') as f:
            self.states_value = pickle.load(f)

class RandomAgent(Agent):
    def move(self, game):
        empty_cells = game.get_empty_positions()
        if not empty_cells:
            raise ValueError("No more moves left to play.")
        # Select a random move from the list of empty cells
        return empty_cells[np.random.randint(len(empty_cells))]

class AggressiveAgent(Agent):
    def __init__(self, symbol):
        super().__init__(symbol)

    def move(self, game):
        empty_positions = game.get_empty_positions()
        board_copy = game.board.copy()
        for position in empty_positions:
            board_copy[position] = self.symbol
            if game.is_winner(self):
                return position
            board_copy[position] = EMPTY  # Reset the position after check

        # If no winning move found, return a random move
        return empty_positions[np.random.choice(len(empty_positions))]

class DefensiveAgent(Agent):
    def __init__(self, symbol):
        super().__init__(symbol)

    def move(self, game):
        opponent_symbol = PLAYER_O if self.symbol == PLAYER_X else PLAYER_X
        empty_positions = game.get_empty_positions()
        board_copy = game.board.copy()

        # First, check if the opponent has a winning move and block it
        for position in empty_positions:
            board_copy[position] = opponent_symbol
            if game.is_winner(self.__opponent()):
                return position  # Block the opponent's winning move
            board_copy[position] = EMPTY  # Reset the position after check

        # If no blocking move is necessary, choose a random move
        return empty_positions[np.random.choice(len(empty_positions))]

    def __opponent(self):
        # Private helper method to create a 'dummy' opponent with the opposite symbol
        return RandomAgent(PLAYER_O if self.symbol == PLAYER_X else PLAYER_X)

#### 0.3: Useful and example functions:
Some may never been called

In [ ]:
def get_hash(board=None):
    return ','.join(str(int(elem)) for elem in board.flatten())


def get_hashes(boards):
    return [get_hash(board) for board in boards]

def valid_state(board, symbol):

    # check the board state is valid
    if symbol==PLAYER_X:
        return (np.sum(board==PLAYER_X)==np.sum(board==PLAYER_O))
    if symbol==PLAYER_O:
        return (np.sum(board==PLAYER_X)-np.sum(board==PLAYER_O)==1)

def next_symbol(board):
    if (np.sum(board==PLAYER_X)==np.sum(board==PLAYER_O)):
        return PLAYER_X
    else:
        return PLAYER_O

def is_terminal(board):
    # Check for a win in rows, columns, and diagonals
    for i in range(GAME_ROW):
        if abs(np.sum(board[i, :])) == 3 or abs(np.sum(board[:, i])) == 3:
            return True
    if abs(sum(np.diag(board))) == 3 or abs(sum(np.diag(np.fliplr(board)))) == 3:
        return True
    # Check for a draw
    if not np.any(board == EMPTY):
        return True
    return False

def get_reward(board, symbol):
    # Define opponent's symbol
    opponent_symbol = PLAYER_O if symbol == PLAYER_X else PLAYER_X

    # Check for current player's win
    for i in range(GAME_ROW):
        if sum(board[i, :]) == GAME_ROW * symbol or sum(board[:, i]) == GAME_COL * symbol:
            return 1
    if sum(np.diag(board)) == GAME_ROW * symbol or sum(np.diag(np.fliplr(board))) == GAME_COL * symbol:
        return 1

    # Check for opponent's win
    for i in range(GAME_ROW):
        if sum(board[i, :]) == GAME_ROW * opponent_symbol or sum(board[:, i]) == GAME_COL * opponent_symbol:
            return -1
    if sum(np.diag(board)) == GAME_ROW * opponent_symbol or sum(np.diag(np.fliplr(board))) == GAME_COL * opponent_symbol:
        return -1

    # Check for a draw
    if is_terminal(board):
        return 0

    # For non-terminal states, the immediate reward is 0.
    return 0


def get_empty_positions(board):
    return [(i, j) for i in range(GAME_ROW) for j in range(GAME_COL) if board[i, j] == EMPTY]

def generate_next_boardstates(board, symbol):

    # check the board state is valid
    if symbol==PLAYER_X:
        assert(np.sum(board==PLAYER_X)==np.sum(board==PLAYER_O))
    if symbol==PLAYER_O:
        assert(np.sum(board==PLAYER_X)-np.sum(board==PLAYER_O)==1)

    # generate all next board states
    all_empty_positions = get_empty_positions(board)
    all_boards = np.tile(board, (len(all_empty_positions), 1, 1))
    all_indices = np.concatenate(
        [
            np.expand_dims(np.arange(len(all_empty_positions)), axis=1),
            np.array(all_empty_positions)
        ], axis=1
        )
    all_boards[all_indices[:,0], all_indices[:,1], all_indices[:,2]] = symbol
    all_boards = np.split(all_boards, all_boards.shape[0], axis=0)
    return [np.squeeze(board) for board in all_boards]

def generate_all_states(board, all_states=None, stop_step=None):

    # assert(valid_state(board, symbol)) # validate the states and symbol

    if all_states is None:
        # all_states = {}
        boards = [board]
        state_hashes = [get_hash(board)]
        p0 = 0
        p1 = 1
        p2 = p1
        step = 0
        step_symbol = next_symbol(board)

    while p0!=p1:
      for p_state in range(p0,p1):
          # print(step)
          # print('p_state:', p_state)
          # print('board:', boards[p_state])
          # print('step_symbol:', step_symbol)
          # print(p1-p0)
          # print('------------------------')
          if is_terminal(boards[p_state]):
              continue
          next_boards = generate_next_boardstates(boards[p_state], step_symbol)
          next_hashes = get_hashes(next_boards)

          # print(next_boards)

          boards+=next_boards
          state_hashes+=next_hashes
          p2 += len(next_boards)

      step_symbol = PLAYER_X if step_symbol == PLAYER_O else PLAYER_O
      p0 = p1
      p1 = p2
      step+=1

      if stop_step is not None:
        if step == stop_step:
          break
    return dict(zip(state_hashes, boards))

#### 0.4: Generate all states using the function **generate_all_states** and save the states hush table in the local path for the future use.

In [ ]:
state_hush_fname = "all_states_hush1.txt"
if os.path.isfile(state_hush_fname):
    with open(state_hush_fname, 'rb') as f:
        all_states = pickle.load(f)

else:
  temp_board = np.zeros((GAME_ROW, GAME_COL))
  all_states = generate_all_states(temp_board)

  # same hush table of all the states
  with open(state_hush_fname, 'wb') as f:
      pickle.dump(all_states, f)

print("In total, ", len(all_states), "states")

In total,  5478 states


---
---

### <font color="blue"> **Task 1 (7 marks)**:</font> Value Iteration

#### <font color="blue"> **Question 1:** </font> Write a value iteration agent in ValueIterationAgent which has been partially specified for you. Here you need to implement the train() & get_reward() methods. The former should perform **planning* using *value iteration and the latter should extract the policy and compute state values.

In [ ]:
class ValueIterationAgent(Agent):
    def __init__(self, symbol, discount_factor=0.9, living_reward=-0.01):
        super().__init__(symbol)
        if 'all_states' in globals():
            self.all_states = all_states
        elif os.path.isfile(state_hush_fname):
            with open(state_hush_fname, 'rb') as f:
                self.all_states = pickle.load(f)
        else:
            raise Exception("No state hushes! Either run the code by order or create the state hush yourself")

        self.discount_factor = discount_factor  # Discount factor for future rewards
        self.living_reward = living_reward  # Reward for living (negative for penalty)
        self.value_function = {state: 0 for state in self.all_states.keys()}  # Initialize state values to 0

        self.win_reward = 10.0
        self.lose_reward = -50.0
        self.living_reward = -1.0
        self.draw_reward = 0.0

        self.policy = {}  # Initialize policy

    def get_reward(self, board, symbol):
        # Define opponent's symbol
        opponent_symbol = PLAYER_O if symbol == PLAYER_X else PLAYER_X

        # Check for current player's win
        for i in range(GAME_ROW):
            if sum(board[i, :]) == GAME_ROW * symbol or sum(board[:, i]) == GAME_COL * symbol:
                return self.win_reward
        if sum(np.diag(board)) == GAME_ROW * symbol or sum(np.diag(np.fliplr(board))) == GAME_COL * symbol:
            return self.win_reward

        # Check for opponent's win
        for i in range(GAME_ROW):
            if sum(board[i, :]) == GAME_ROW * opponent_symbol or sum(board[:, i]) == GAME_COL * opponent_symbol:
                return self.lose_reward
        if sum(np.diag(board)) == GAME_ROW * opponent_symbol or sum(np.diag(np.fliplr(board))) == GAME_COL * opponent_symbol:
            return self.lose_reward

        # Check for a draw
        if np.all(board != EMPTY):
            return self.draw_reward

        # For non-terminal states, the immediate reward is 0.
        return 0

    def train(self, threshold=0.00001):
        # Value iteration algorithm
        while True:
            delta = 0
            for state_hash, board in self.all_states.items():
                # Create a game instance with a random agent for both players
                game_instance = Game(RandomAgent(PLAYER_X), RandomAgent(PLAYER_O))
                game_instance.board = board

                # Check if the game is not in a terminal state
                if not game_instance.is_terminal():
                    old_value = self.value_function[state_hash]
                    possible_moves = game_instance.get_empty_positions()

                    # Initialize a list to store the values of each possible move
                    move_values = [self.get_reward(game_instance.board, self.symbol)]
                    for move in possible_moves:
                        next_board = game_instance.board.copy()
                        next_board[move] = self.symbol
                        next_state_hash = game_instance.get_hash(next_board)


                        if next_state_hash not in self.value_function:
                            self.value_function[next_state_hash] = 0

                        move_values.append(self.discount_factor * self.value_function[next_state_hash])

                    # Update the value of the current state to the maximum value among possible moves
                    self.value_function[state_hash] = max(move_values)
                    delta = max(delta, abs(old_value - self.value_function[state_hash]))

            if delta < threshold:
                break

        for state_hash, board in self.all_states.items():
            game_instance = Game(RandomAgent(PLAYER_X), RandomAgent(PLAYER_O))
            game_instance.board = board

            # Check if the game is not in a terminal state
            if not game_instance.is_terminal():
                next_board_values = []
                possible_moves = game_instance.get_empty_positions()
                # Iterate over possible moves to find the best move
                for move in possible_moves:
                    next_board = game_instance.board.copy()
                    next_board[move] = self.symbol
                    next_state_hash = game_instance.get_hash(next_board)

                    # Ensure the next state is in the value function, if not, initialize it
                    if next_state_hash not in self.value_function:
                        self.value_function[next_state_hash] = 0

                    next_board_values.append(self.value_function[next_state_hash])

                # If there are valid moves, select the best move and update the policy
                if next_board_values:
                    best_move = possible_moves[np.argmax(next_board_values)]
                    self.policy[state_hash] = best_move

    def move(self, game):
        # Return the move based on the current policy
        current_state = game.get_hash()
        if current_state in self.policy:
            return self.policy[current_state]
        else:
            # In case the current state is not in the policy, choose a random move
            empty_positions = game.get_empty_positions()
            return empty_positions[np.random.choice(len(empty_positions))]


#### <font color="blue">Q1.1 (3/7): Run the following example of a Value Iteration "X" player against a Random "O" agent </font>




In [ ]:
player_x = ValueIterationAgent(PLAYER_X)  # This is the value iteration agent
player_o = RandomAgent(PLAYER_O)  # This is the random agent

game = Game(player_x, player_o)
# print(game.board)

# Compute the policy using value iteration only for the value iteration agent
player_x.train()  # We only need to compute this for player O

# Play the game

game.play()

Player 1 wins!


#### <font color="blue"> Q1.2 (3/7): Based on the example (Game 0: RandomAgent "O" v.s. ValueIterationAgent "X") above, run the following game: </font>
* Game1: RandomAgent "X" v.s. ValueIterationAgent "O"
* Game2: ValueIterationAgent "X" v.s. AggressiveAgent "O"
* Game3: ValueIterationAgent "O" v.s. AggressiveAgent "X"
* Game4: ValueIterationAgent "X" v.s. DefensiveAgent "O"
* Game 5: ValueIterationAgent "O" v.s. DefensiveAgent "X"

Use the one single code cell *below*

In [ ]:
# Game 1:
player_x1 = RandomAgent(PLAYER_X)
player_o1 = ValueIterationAgent(PLAYER_O)

game1 = Game(player_x1, player_o1)
game1.play()

# Game 2:
player_x2 = ValueIterationAgent(PLAYER_X)
player_o2 = AggressiveAgent(PLAYER_O)

game2 = Game(player_x2, player_o2)
game2.play()

# Game 3:
player_x3 = AggressiveAgent(PLAYER_X)
player_o3 = ValueIterationAgent(PLAYER_O)

game3 = Game(player_x3, player_o3)
game3.play()

# Game 4:
player_x4 = ValueIterationAgent(PLAYER_X)
player_o4 = DefensiveAgent(PLAYER_O)

game4 = Game(player_x4, player_o4)
game4.play()

# Game 5:
player_x5 = DefensiveAgent(PLAYER_X)
player_o5 = ValueIterationAgent(PLAYER_O)

game5 = Game(player_x5, player_o5)
game5.play()


Player 1 wins!
Player 1 wins!
Player -1 wins!
Player -1 wins!
Player 1 wins!


#### <font color="blue"> Q1.3 (1/7): Repeat the games (Game 0-5) above 50 rounds each Game. Using ValueIterationAgent, print out number of *wins*, *losts* and *draw* </font>

In [ ]:
# Q1.2 1-5:

game.show_board = False # Disable printing board, don't change

# Function to play multiple rounds of a given game
def play_multiple_rounds(player_x, player_o, rounds):
    wins_x = 0
    wins_o = 0
    draws = 0

    for _ in range(rounds):
        game = Game(player_x, player_o)
        game.play()

        if game.winner == PLAYER_X:
            wins_x += 1
        elif game.winner == PLAYER_O:
            wins_o += 1
        else:
            draws += 1

    return wins_x, wins_o, draws

# Set the number of rounds
rounds = 50

# Game 1:
wins_x1, wins_o1, draws1 = play_multiple_rounds(player_x1, player_o1, rounds)
print("Game 1 - RandomAgent 'X' vs ValueIterationAgent 'O':")
print(f"Wins X: {wins_x1}, Wins O: {wins_o1}, Draws: {draws1}\n")

# Game 2:
wins_x2, wins_o2, draws2 = play_multiple_rounds(player_x2, player_o2, rounds)
print("\nGame 2 - ValueIterationAgent 'X' vs AggressiveAgent 'O':")
print(f"Wins X: {wins_x2}, Wins O: {wins_o2}, Draws: {draws2}\n")

# Game 3:
wins_x3, wins_o3, draws3 = play_multiple_rounds(player_x3, player_o3, rounds)
print("\nGame 3 - AggressiveAgent 'X' vs ValueIterationAgent 'O':")
print(f"Wins X: {wins_x3}, Wins O: {wins_o3}, Draws: {draws3}\n")

# Game 4:
wins_x4, wins_o4, draws4 = play_multiple_rounds(player_x4, player_o4, rounds)
print("\nGame 4 - ValueIterationAgent 'X' vs DefensiveAgent 'O':")
print(f"Wins X: {wins_x4}, Wins O: {wins_o4}, Draws: {draws4}\n")

# Game 5:
wins_x5, wins_o5, draws5 = play_multiple_rounds(player_x5, player_o5, rounds)
print("\nGame 5 - DefensiveAgent 'X' vs ValueIterationAgent 'O':")
print(f"Wins X: {wins_x5}, Wins O: {wins_o5}, Draws: {draws5}\n")


Player 1 wins!
It's a draw!
Player 1 wins!
It's a draw!
Player -1 wins!
Player -1 wins!
Player 1 wins!
Player 1 wins!
It's a draw!
Player -1 wins!
Player -1 wins!
Player 1 wins!
Player -1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player -1 wins!
It's a draw!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player -1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player -1 wins!
Player -1 wins!
Player 1 wins!
Player 1 wins!
Player -1 wins!
Player 1 wins!
Player -1 wins!
Player -1 wins!
Player 1 wins!
It's a draw!
Player -1 wins!
Player 1 wins!
Game 1 - RandomAgent 'X' vs ValueIterationAgent 'O':
Wins X: 32, Wins O: 13, Draws: 5

Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player -1 wins!
Player 1 wins!
It's a draw!
Player -1 wins!
Player 1 w

---
---

### <font color="blue"> **Task 2** (7 marks):</font>  Policy Iteration

Write a Policy Iteration agent in PolicyIterationAgent by implementing the policy_evaluation(), policy_improvement(), train() methods. The policy_evaluation() method should evaluate the current policy (see your lecture notes). The current values for the current policy should be stored in the provided policyValues map. The policy_improvement() method performs the Policy improvement step, and updates curPolicy. The train() method is the planning process, once done, an optimal policy should be saved in the agent object.

In [ ]:
import random

class PolicyIterationAgent(Agent):
    def __init__(self, symbol, discount_factor=0.9, living_reward=-0.01):
        super().__init__(symbol)

        if 'all_states' in globals():
            self.all_states = all_states
        elif os.path.isfile(state_hush_fname):
            with open(state_hush_fname, 'rb') as f:
                self.all_states = pickle.load(f)
        else:
            raise Exception("No state hushes! Either run the code by order or create the state hush yourself")

        self.discount_factor = discount_factor  # Discount factor for future rewards
        self.living_reward = living_reward  # Reward for living (negative for penalty)
        self.value_function = {state: 0 for state in all_states.keys()}  # Initialize state values to 0

        # self.all_states = all_states  # All possible states
        self.win_reward=10.0;
        self.lose_reward=-50.0;
        self.living_reward=-1.00;
        self.draw_reward=0.0;

        self.all_states = all_states  # All possible states
        self.discount_factor = discount_factor  # Discount factor for future rewards
        # self.living_reward = living_reward  # Living reward (negative for penalty)
        self.value_function = {state: 0 for state in all_states.keys()}  # Initialize state values to 0
        # self.policy = {state: np.random.choice(get_empty_positions(board))
        #                for state, board in all_states.items() if not is_terminal(board)}  # Random initial policy
        # # Choosing a random tuple from the list of empty positions
        self.policy = {state: random.choice(get_empty_positions(board))
                   for state, board in all_states.items() if not is_terminal(board)}  # Random initial policy

    def get_reward(self, board, symbol):
        # Define opponent's symbol
        opponent_symbol = PLAYER_O if symbol == PLAYER_X else PLAYER_X

        # Check for current player's win
        for i in range(GAME_ROW):
            if sum(board[i, :]) == GAME_ROW * symbol or sum(board[:, i]) == GAME_COL * symbol:
                return self.win_reward
        if sum(np.diag(board)) == GAME_ROW * symbol or sum(np.diag(np.fliplr(board))) == GAME_COL * symbol:
            return self.win_reward

        # Check for opponent's win
        for i in range(GAME_ROW):
            if sum(board[i, :]) == GAME_ROW * opponent_symbol or sum(board[:, i]) == GAME_COL * opponent_symbol:
                return self.lose_reward
        if sum(np.diag(board)) == GAME_ROW * opponent_symbol or sum(np.diag(np.fliplr(board))) == GAME_COL * opponent_symbol:
            return self.lose_reward

        # Check for a draw
        if np.all(board != EMPTY):
            return self.draw_reward

        # For non-terminal states, the immediate reward is 0.
        return 0

    def policy_evaluation(self, threshold=0.0001):
        while True:
            delta = 0
            for state_hash, board in self.all_states.items():
                if not is_terminal(board):
                    old_value = self.value_function[state_hash]
                    current_move = self.policy[state_hash]

                    next_board = board.copy()
                    next_board[current_move] = self.symbol
                    next_state_hash = get_hash(next_board)

                    # Add the new state to the value function if it's not present
                    if next_state_hash not in self.value_function:
                        self.value_function[next_state_hash] = 0

                    # Update the value function using the Bellman equation
                    self.value_function[state_hash] = self.get_reward(board, self.symbol) + \
                                                       self.discount_factor * self.value_function[next_state_hash]

                    delta = max(delta, abs(old_value - self.value_function[state_hash]))

            if delta < threshold:
                break

    def policy_improvement(self):
        policy_stable = True

        for state_hash, board in self.all_states.items():
            if not is_terminal(board):
                old_move = self.policy[state_hash]
                possible_moves = get_empty_positions(board)

                move_values = {move: self.get_reward(board, self.symbol) +
                                      self.discount_factor * self.value_function[get_hash(board)] for move in possible_moves}

                # Update the policy by selecting the move with the maximum value
                self.policy[state_hash] = max(move_values, key=move_values.get)

                if old_move != self.policy[state_hash]:
                    policy_stable = False

        return policy_stable

    def train(self):
        while True:
            self.policy_evaluation()
            if self.policy_improvement():
                break

    def move(self, game):
        # Return the move based on the current policy
        current_state = game.get_hash()
        if current_state in self.policy:
            return self.policy[current_state]
        else:
            # If the current state is not in the policy, choose a random move
            empty_positions = game.get_empty_positions()
            return empty_positions[np.random.choice(len(empty_positions))]


#### <font color="blue">Q2.1 (3/7): Run the following: Iteration "X" player against a Random "O" agent </font>

In [ ]:
player_o = RandomAgent(PLAYER_O)  # This is the random agent
player_x = PolicyIterationAgent(PLAYER_X)  # This is the value iteration agent

# player_x = ValueIterationAgent(PLAYER_X)  # This is the random agent
# player_o = RandomAgent(PLAYER_O)  # This is the value iteration agent

game = Game(player_x, player_o)
# print(game.board)

# Compute the policy using value iteration only for the value iteration agent
player_x.train()  # We only need to compute this for player O

# Play the game

game.play()

Player -1 wins!


#### <font color="blue"> Q2.2 (3/7): Based on the example (Game 0: RandomAgent "O" v.s. PlicyIterationAgent "X") above, run the following game: </font>
* Game1: RandomAgent "X" v.s. PolicyIterationAgent "O"
* Game2: PolicyIterationAgent "X" v.s. AggressiveAgent "O"
* Game3: PolicyIterationAgent "O" v.s. AggressiveAgent "X"
* Game4: PolicyIterationAgent "X" v.s. DefensiveAgent "O"
* Game 5: PolicyIterationAgent "O" v.s. DefensiveAgent "X"

Use the one single code cell *below*

In [ ]:
# Game 1:
player_x1 = RandomAgent(PLAYER_X)
player_o1 = PolicyIterationAgent(PLAYER_O)

game1 = Game(player_x1, player_o1)
game1.play()

# Game 2:
player_x2 = PolicyIterationAgent(PLAYER_X)
player_o2 = AggressiveAgent(PLAYER_O)

game2 = Game(player_x2, player_o2)
game2.play()

# Game 3:
player_x3 = AggressiveAgent(PLAYER_X)
player_o3 = PolicyIterationAgent(PLAYER_O)

game3 = Game(player_x3, player_o3)
game3.play()

# Game 4:
player_x4 = PolicyIterationAgent(PLAYER_X)
player_o4 = DefensiveAgent(PLAYER_O)

game4 = Game(player_x4, player_o4)
game4.play()

# Game 5:
player_x5 = DefensiveAgent(PLAYER_X)
player_o5 = PolicyIterationAgent(PLAYER_O)

game5 = Game(player_x5, player_o5)
game5.play()

It's a draw!
Player 1 wins!
It's a draw!
Player 1 wins!
It's a draw!


#### <font color="blue"> Q2.3 (1/7): Repeat the games (Game 0-5) above 50 rounds each Game. Using PolicyIterationAgent, print out number of *wins*, *losts* and *draw* </font>

In [ ]:
game.show_board = False

# Define a function to play multiple rounds of a given game
def play_multiple_rounds(player_x, player_o, rounds):
    wins_x = 0
    wins_o = 0
    draws = 0

    for _ in range(rounds):
        game = Game(player_x, player_o)
        game.play()

        if game.winner == PLAYER_X:
            wins_x += 1
        elif game.winner == PLAYER_O:
            wins_o += 1
        else:
            draws += 1

    return wins_x, wins_o, draws


# Set the number of rounds
rounds = 50

# Game 0: RandomAgent 'O' vs. PolicyIterationAgent 'X'
wins_x0, wins_o0, draws0 = play_multiple_rounds(PolicyIterationAgent(PLAYER_X), RandomAgent(PLAYER_O), rounds)
print("Game 0 - RandomAgent 'O' vs. PolicyIterationAgent 'X':")
print(f"Wins X: {wins_x0}, Wins O: {wins_o0}, Draws: {draws0}\n")

# Game 1: RandomAgent 'X' vs. PolicyIterationAgent 'O'
wins_x1, wins_o1, draws1 = play_multiple_rounds(RandomAgent(PLAYER_X), PolicyIterationAgent(PLAYER_O), rounds)
print("\nGame 1 - RandomAgent 'X' vs. PolicyIterationAgent 'O':")
print(f"Wins X: {wins_x1}, Wins O: {wins_o1}, Draws: {draws1}\n")

# Game 2: PolicyIterationAgent 'X' vs. AggressiveAgent 'O'
wins_x2, wins_o2, draws2 = play_multiple_rounds(PolicyIterationAgent(PLAYER_X), AggressiveAgent(PLAYER_O), rounds)
print("\nGame 2 - PolicyIterationAgent 'X' vs. AggressiveAgent 'O':")
print(f"Wins X: {wins_x2}, Wins O: {wins_o2}, Draws: {draws2}\n")

# Game 3: PolicyIterationAgent 'O' vs. AggressiveAgent 'X'
wins_x3, wins_o3, draws3 = play_multiple_rounds(AggressiveAgent(PLAYER_X), PolicyIterationAgent(PLAYER_O), rounds)
print("\nGame 3 - AggressiveAgent 'X' vs. PolicyIterationAgent 'O':")
print(f"Wins X: {wins_x3}, Wins O: {wins_o3}, Draws: {draws3}\n")

# Game 4: PolicyIterationAgent 'X' vs. DefensiveAgent 'O'
wins_x4, wins_o4, draws4 = play_multiple_rounds(PolicyIterationAgent(PLAYER_X), DefensiveAgent(PLAYER_O), rounds)
print("\nGame 4 - PolicyIterationAgent 'X' vs. DefensiveAgent 'O':")
print(f"Wins X: {wins_x4}, Wins O: {wins_o4}, Draws: {draws4}\n")

# Game 5: PolicyIterationAgent 'O' vs. DefensiveAgent 'X'
wins_x5, wins_o5, draws5 = play_multiple_rounds(DefensiveAgent(PLAYER_X), PolicyIterationAgent(PLAYER_O), rounds)
print("\nGame 5 - DefensiveAgent 'X' vs. PolicyIterationAgent 'O':")
print(f"Wins X: {wins_x5}, Wins O: {wins_o5}, Draws: {draws5}\n")


Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player -1 wins!
Player -1 wins!
Player 1 wins!
Player -1 wins!
Player -1 wins!
Player -1 wins!
Player 1 wins!
Player -1 wins!
Player -1 wins!
Player -1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player -1 wins!
It's a draw!
Player 1 wins!
Player -1 wins!
Player 1 wins!
Player 1 wins!
Player -1 wins!
Player 1 wins!
Player -1 wins!
Player 1 wins!
Player -1 wins!
Player 1 wins!
Player -1 wins!
Player 1 wins!
It's a draw!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player -1 wins!
Player 1 wins!
It's a draw!
Player -1 wins!
Player 1 wins!
It's a draw!
Player 1 wins!
Player 1 wins!
It's a draw!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Game 0 - RandomAgent 'O' vs. PolicyIterationAgent 'X':
Wins X: 29, Wins O: 16, Draws: 5

Player -1 wins!
Player -1 wins!
Player 1 wins!
Player 1 wins!
It's a draw!
Player 1 wins!
Player -1 wins!
Player 1 wins!
Player -1 wins!
Player -1 wins!
Pl

### <font color="blue"> **Task 3** (6 marks):</font>  Q-Learn

Write a QLearn agent in QLearnIterationAgent. No specific requirements of functions, but the planning process have to be done in a plan() function.

In [ ]:
class QLearningAgent(Agent):
    def __init__(self, symbol, alpha=0.4, gamma=0.9, epsilon=0.1, living_penalty=-1):
        super().__init__(symbol)
        self.alpha = alpha  # Learning rate
        self.gamma = gamma  # Discount factor
        self.epsilon = epsilon  # Epsilon for the epsilon-greedy policy
        self.Q = {}  # Initialize Q-table
        self.living_penalty = living_penalty
        self.win_reward = 10.0
        self.lose_reward = -50.0
        self.draw_reward = 0.0
        self.initial_state = np.zeros((GAME_ROW, GAME_COL), dtype=int)  # Initialize the initial state
        self.current_symbol = symbol  # The symbol of the current player

    def hash_state(self, state):
        return str(state.reshape(GAME_ROW * GAME_COL))

    def get_available_actions(self, state):
        return [(i, j) for i in range(GAME_ROW) for j in range(GAME_COL) if state[i, j] == EMPTY]

    def choose_action(self, state_hash, available_actions):
        if not available_actions:
            # If there are no available actions, return None
            return None

        if np.random.rand() < self.epsilon:
            # Exploration: choose a random action
            return available_actions[np.random.choice(len(available_actions))]
        else:
            # Exploitation: choose the action with the highest Q-value
            if state_hash not in self.Q:
                # If the state is not in the Q-table, initialize it with zeros
                self.Q[state_hash] = {action: 0.0 for action in available_actions}
            return max(self.Q[state_hash], key=self.Q[state_hash].get)


    def plan(self, state, action, next_state, reward, done):
        # Q-learning update rule
        state_hash = self.hash_state(state)
        next_state_hash = self.hash_state(next_state)

        if state_hash not in self.Q:
            self.Q[state_hash] = {action: 0.0 for action in self.get_available_actions(state)}

        if next_state_hash not in self.Q:
            self.Q[next_state_hash] = {action: 0.0 for action in self.get_available_actions(next_state)}

        # Update Q-value for the current state-action pair
        if self.Q[next_state_hash]:
            max_q_value = max(self.Q[next_state_hash].values())
        else:
            max_q_value = 0.0

        self.Q[state_hash][action] += self.alpha * (
            reward + self.gamma * max_q_value - self.Q[state_hash][action])


    def train(self, num_episodes=100000):
        for _ in range(num_episodes):
            game = Game(self, RandomAgent(PLAYER_O))
            state = np.zeros((GAME_ROW, GAME_COL), dtype=int)
            done = False

            while not done:
                available_actions = self.get_available_actions(state)
                action = self.choose_action(self.hash_state(state), available_actions)

                if action is not None:  # Check if action is not None
                    next_state, reward, done = self.make_move(state, action)
                    self.plan(state, action, next_state, reward, done)
                    state = next_state

    def get_reward(self, state, symbol):
        # Define opponent's symbol
        opponent_symbol = PLAYER_O if symbol == PLAYER_X else PLAYER_X

        # Check for current player's win
        for i in range(GAME_ROW):
            if sum(state[i, :]) == GAME_ROW * symbol or sum(state[:, i]) == GAME_COL * symbol:
                return self.win_reward

        if sum(np.diag(state)) == GAME_ROW * symbol or sum(np.diag(np.fliplr(state))) == GAME_COL * symbol:
            return self.win_reward

        # Check for opponent's win
        for i in range(GAME_ROW):
            if sum(state[i, :]) == GAME_ROW * opponent_symbol or sum(state[:, i]) == GAME_COL * opponent_symbol:
                return self.lose_reward

        if sum(np.diag(state)) == GAME_ROW * opponent_symbol or sum(np.diag(np.fliplr(state))) == GAME_COL * opponent_symbol:
            return self.lose_reward

        # Check for a draw
        if is_terminal(state):
            return self.draw_reward

        # For non-terminal states, the immediate reward is 0.
        return 0

    def make_move(self, state, action):
        new_state = np.array(state)
        new_state[action] = self.current_symbol
        reward = self.get_reward(new_state, self.current_symbol)
        done = is_terminal(new_state)
        return new_state, reward, done

    def move(self, game):
        # Extract the board from the Game object
        board = game.board
        state_hash = self.hash_state(board)
        available_actions = self.get_available_actions(board)

        if not available_actions:
            raise ValueError("No available actions to make a move.")

        # Choose the best action based on the Q-table
        action = self.choose_action(state_hash, available_actions)

        # Convert action to the format expected by Game's make_move method (e.g., (row, col))
        return action

#### <font color="blue">Q3.1 (3/6): Run the following example: Iteration "X" player against a Random "O" agent </font>

In [ ]:
# Initialize the QLearningAgent with its symbol (X or O)
q_learning_agent = QLearningAgent(PLAYER_X)

# Assume there is a random agent for the opponent
random_agent = RandomAgent(PLAYER_O)

# Initialize the game environment with both agents
game = Game(q_learning_agent, random_agent)

# Train the QLearningAgent with a function that simulates playing the game
# The train function would need to be implemented to simulate games within the agent
q_learning_agent.train()

# Use the game's play function to start playing
game.play()


Player 1 wins!


#### <font color="blue"> Q3.2 (2/6): Based on the example (Game 0: RandomAgent "O" v.s. QLearnAgent "X") above, run the following game: </font>
* Game1: RandomAgent "X" v.s. QLearnAgent "O"
* Game2: QLearnAgent "X" v.s. AggressiveAgent "O"
* Game3: QLearnAgent "O" v.s. AggressiveAgent "X"
* Game4: QLearnAgent "X" v.s. DefensiveAgent "O"
* Game 5: QLearnAgent "O" v.s. DefensiveAgent "X"

Use the one single code cell *below*

In [ ]:
# Game 1:
random_agent_x = RandomAgent(PLAYER_X)
qlearn_agent_o = QLearningAgent(PLAYER_O)

game1 = Game(random_agent_x, qlearn_agent_o, show_board=True)
game1.play()

# Game 2:
qlearn_agent_x = QLearningAgent(PLAYER_X)
aggressive_agent_o = AggressiveAgent(PLAYER_O)

game2 = Game(qlearn_agent_x, aggressive_agent_o, show_board=True)
game2.play()


# Game 3:
qlearn_agent_o = QLearningAgent(PLAYER_O)
aggressive_agent_x = AggressiveAgent(PLAYER_X)

game3 = Game(qlearn_agent_o, aggressive_agent_x, show_board=True)
game3.play()

# Game 4:
qlearn_agent_x = QLearningAgent(PLAYER_X)
defensive_agent_o = DefensiveAgent(PLAYER_O)

game4 = Game(qlearn_agent_x, defensive_agent_o, show_board=True)
game4.play()

# Game 5:
qlearn_agent_o = QLearningAgent(PLAYER_O)
defensive_agent_x = DefensiveAgent(PLAYER_X)

game5 = Game(qlearn_agent_o, defensive_agent_x, show_board=True)
game5.play()


Turn 1: Player 1
| |X| |
| | | |
| | | |

Turn 2: Player -1
|O|X| |
| | | |
| | | |

Turn 3: Player 1
|O|X| |
| | | |
| |X| |

Turn 4: Player -1
|O|X|O|
| | | |
| |X| |

Turn 5: Player 1
|O|X|O|
| | | |
|X|X| |

Turn 6: Player -1
|O|X|O|
|O| | |
|X|X| |

Turn 7: Player 1
|O|X|O|
|O| |X|
|X|X| |

Turn 8: Player -1
|O|X|O|
|O|O|X|
|X|X| |

Player 1 wins!
|O|X|O|
|O|O|X|
|X|X|X|

Turn 1: Player 1
|X| | |
| | | |
| | | |

Turn 2: Player -1
|X| | |
|O| | |
| | | |

Turn 3: Player 1
|X|X| |
|O| | |
| | | |

Turn 4: Player -1
|X|X| |
|O| |O|
| | | |

Player 1 wins!
|X|X|X|
|O| |O|
| | | |

Turn 1: Player -1
|O| | |
| | | |
| | | |

Turn 2: Player 1
|O| | |
| |X| |
| | | |

Turn 3: Player -1
|O| | |
| |X| |
| | |O|

Turn 4: Player 1
|O|X| |
| |X| |
| | |O|

Turn 5: Player -1
|O|X|O|
| |X| |
| | |O|

Turn 6: Player 1
|O|X|O|
|X|X| |
| | |O|

Player -1 wins!
|O|X|O|
|X|X|O|
| | |O|

Turn 1: Player 1
|X| | |
| | | |
| | | |

Turn 2: Player -1
|X| | |
| | | |
|O| | |

Turn 3: Player 1
|X|X| |
| | 

#### <font color="blue"> Q3.3 (1/7): Repeat the games (Game 0-5) above 50 rounds each Game. Using QLearnAgent, print out number of *wins*, *losts* and *draw* </font>

In [ ]:
game.show_board = False

def run_game(agent1, agent2, rounds=50):
    wins = 0
    losses = 0
    draws = 0

    for _ in range(rounds):
        game = Game(agent1, agent2, show_board=False)
        game.play()

        if game.winner == agent1.symbol:
            wins += 1
        elif game.winner == agent2.symbol:
            losses += 1
        else:
            draws += 1

    return wins, losses, draws

# Repeat Game 0 (RandomAgent "O" v.s. QLearnAgent "X") 50 times
random_agent_o = RandomAgent(PLAYER_O)
qlearn_agent_x = QLearningAgent(PLAYER_X)
game0_results = run_game(random_agent_o, qlearn_agent_x)

# Repeat Game 1 (RandomAgent "X" v.s. QLearnAgent "O") 50 times
random_agent_x = RandomAgent(PLAYER_X)
qlearn_agent_o = QLearningAgent(PLAYER_O)
game1_results = run_game(random_agent_x, qlearn_agent_o)

# Repeat Game 2 (QLearnAgent "X" v.s. AggressiveAgent "O") 50 times
qlearn_agent_x = QLearningAgent(PLAYER_X)
aggressive_agent_o = AggressiveAgent(PLAYER_O)
game2_results = run_game(qlearn_agent_x, aggressive_agent_o)

# Repeat Game 3 (QLearnAgent "O" v.s. AggressiveAgent "X") 50 times
qlearn_agent_o = QLearningAgent(PLAYER_O)
aggressive_agent_x = AggressiveAgent(PLAYER_X)
game3_results = run_game(qlearn_agent_o, aggressive_agent_x)

# Repeat Game 4 (QLearnAgent "X" v.s. DefensiveAgent "O") 50 times
qlearn_agent_x = QLearningAgent(PLAYER_X)
defensive_agent_o = DefensiveAgent(PLAYER_O)
game4_results = run_game(qlearn_agent_x, defensive_agent_o)

# Repeat Game 5 (QLearnAgent "O" v.s. DefensiveAgent "X") 50 times
qlearn_agent_o = QLearningAgent(PLAYER_O)
defensive_agent_x = DefensiveAgent(PLAYER_X)
game5_results = run_game(qlearn_agent_o, defensive_agent_x)

# Print the results
print("Game 0 Results (RandomAgent 'O' v.s. QLearnAgent 'X'): Wins =", game0_results[0], "Losses =", game0_results[1], "Draws =", game0_results[2])
print("Game 1 Results (RandomAgent 'X' v.s. QLearnAgent 'O'): Wins =", game1_results[0], "Losses =", game1_results[1], "Draws =", game1_results[2])
print("Game 2 Results (QLearnAgent 'X' v.s. AggressiveAgent 'O'): Wins =", game2_results[0], "Losses =", game2_results[1], "Draws =", game2_results[2])
print("Game 3 Results (QLearnAgent 'O' v.s. AggressiveAgent 'X'): Wins =", game3_results[0], "Losses =", game3_results[1], "Draws =", game3_results[2])
print("Game 4 Results (QLearnAgent 'X' v.s. DefensiveAgent 'O'): Wins =", game4_results[0], "Losses =", game4_results[1], "Draws =", game4_results[2])
print("Game 5 Results (QLearnAgent 'O' v.s. DefensiveAgent 'X'): Wins =", game5_results[0], "Losses =", game5_results[1], "Draws =", game5_results[2])

Player -1 wins!
Player 1 wins!
Player -1 wins!
Player 1 wins!
Player -1 wins!
Player 1 wins!
Player -1 wins!
Player -1 wins!
Player -1 wins!
Player 1 wins!
Player -1 wins!
Player 1 wins!
Player 1 wins!
Player -1 wins!
Player -1 wins!
Player -1 wins!
Player 1 wins!
Player -1 wins!
Player 1 wins!
Player -1 wins!
Player -1 wins!
Player -1 wins!
Player -1 wins!
Player 1 wins!
Player 1 wins!
Player -1 wins!
Player -1 wins!
Player 1 wins!
Player -1 wins!
Player -1 wins!
Player 1 wins!
Player 1 wins!
Player -1 wins!
Player 1 wins!
Player -1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
It's a draw!
Player -1 wins!
Player -1 wins!
Player 1 wins!
Player -1 wins!
Player -1 wins!
Player -1 wins!
Player -1 wins!
Player -1 wins!
Player -1 wins!
It's a draw!
Player -1 wins!
Player 1 wins!
Player 1 wins!
It's a draw!
Player 1 wins!
Player 1 wins!
It's a draw!
Player 1 wins!
Player 1 wins!
Player -1 wins!
Player -1 wins!
Player -1 wins!
Player 1 wins!
Player -1 wins!
Player 1 wins!